# Verständnisdokument zu `min_variance.ipynb`

Dieses Notebook beschäftigt sich mit *min_variance.ipynb*, das NB wird hier step-by-step analysiert.

- Was passiert technisch in jeder Zelle?
- Welche Finance-Idee steckt dahinter?
- Welche Annahmen werden getroffen?
- Welche Teile sind Datenaufbereitung, welche Teile Portfolio-Theorie, welche Teile Evaluation? -> Was steckt hinter jedem Teil?
- Welche offenen Fragen ergeben sich für die Bachelorarbeit?

## 1. Kurzüberblick

**Ziel des Notebooks:**

*Den Prozess der Portfolio Optimisierung mit verschiedenen Möglichkeiten und Wegen aufzeigen*

**Aufbau:**

1. *Data Selection* -> Read the big source and gather a selected universe => Filter for sensful size, no nans
2. *Construct Universe* -> Further select by metric (narrow the universen by a specifc preference)
3. *Estiamte Covariance* -> Two Ways => **sample** & **Ledoit Wolf**
4. *Determine Portfolio Weights* -> different strategies => **naive**, **mcap**, **min_variance**
5. *Form and hold Portfolios* -> Simulate Investors for portfolios
6. *Evaluate Investment* -> compute & plot evaluaten   

**Erste Einordnung:**

| Bereich | Rolle im Notebook | Eigene Notizen |
|---|---|---|
| Daten einlesen | read_feather(); US_data_panel; | Rohe Daten gesamt |
| Daten filtern / vorbereiten | filter_universe(), select_stock(), t_idx TODO | basierend tag t_idx & Datenverfügbarkeit & metrik |
| Portfolio-Gewichte bestimmen | TODO | TODO |
| Rebalancing / Rolling Window | TODO | TODO |
| Performance messen | TODO | TODO |
| Visualisierung | TODO | TODO |

## 2. Cell-for-Cell-Analyse

Für jede Zelle (flexibel):

- **Was macht die Zelle technisch?**
- **Welche Finance- oder Statistik-Idee steckt dahinter?**
- **Welche Inputs werden benötigt?**
- **Welche Outputs entstehen?**
- **Was verstehe ich noch nicht?**

### Zelle 0: Überblick / Titel

**Titel:** `# Overview - Portfolio Optimization Example`

**Was macht diese Zelle?**

Titel & Erklärung des Notebooks. Was wird gemacht (Datenselektion, Gewichtung, Portfoliobildung, Auswertung) 

**Welche Ideen?**

- Data Quality Management (Daten unvollständig o.ä.)
- Multiple Ansätze zur Gewichtung (Naive, macap, minVar) -> Vergleich interessant
- Lookahead & Lookback (basierend auf t, t+ oder t-)

### Zelle 1: Imports und Funktionsdefinitionen

**Titel:** `Methods, Packages and Class Definitions`

**Was macht die Zelle technisch?**

=> Manages Imports and defines the helping structures we need for the actual process

*Bibliotheken:*

- numpy -> Berechnungen
- pandas -> Datenformat -> DataFrame
- sklearn -> Ledoit
- cvxpy -> Optmierung des Miminimerungs Problems
- tqdm -> Ladebalken
- functools -> Partielle Funktionen bzw. Params vorab belegen
- matplotlib -> Datenvisualisierung

*Definitionen:*

(auf Englisch gewechselt, da es irgendwie einfach besser passt)

- <font color="green">filter_universe() </font>

    - takes a panel (pd.DataFrame) and computes how much valid trading days and returns there are
    - if the fraction 1 - #valid_returns / #valid_trading_days is larger than a certain max_missing_frac than the stock is invalid
    - returns all valid sotcks as a List

    - => Filter out all stocks without proper data availability   

- <font color="green">metric_sharpe()</font>

    - computes the pseudo Sharpe-Ratio for a given return_matrix

    - => used for stock selection

- <font color="green">select_stocks()</font>

    - takes a return_matrix, a number_of_stocks and a scoring_metric
    - it then returns a list of the number_of_stocks stocks with the best metric_value

    - => actually selects the stocks 

- <font color="green">panel_to_ret_matrix()</font>
    
    - just reshapes the given panel to a format where we have the stocks as colums and the returns by days as rows

    - => gives us the matrix format we need

- <font color="green">covariance_estimator()</font>

    - takes a estimation method for the covariance matrix
    - computes the covariance matrix
    - makes sure that the diagonal works

    - => gives as the correct Covariance DataFrame

- <font color="green">compute_weights()</font>

    - three methods to compute the portfolio weights (naive, mcap, min_variance)
    - naive is pretty simple
    - mcap gives higher weights to stocks with higher mcaps in the most recent value we have in the estimation window
    - min_variance actually makes the optimisation problem to get the weights accorsing to the risk in the stocks

    - => computes portfolio weights  

- <font color="green">redristribute_weights()</font>

    - if a selected stock shows no data availabilty in the holding period we redistribute the weight it would have gotten to the other stocks in the portfolio

    - => makes sure we remain with our complete capital even when a stocks goes missing for whatever reason

- <font color="green">Investor</font>
    
    - class that wraps our previous logic and is simulate the use of our logic
    - has a strategy (weights)
    - has a estimation method (covariance)
    - has get_returns() to compute portfolio_returns

        - selects the stocks in out portfolio out of the return matrix
        - multiples it with our weights 
        - with matrix multiplication '@' in python
        - => effectively here happens $r_p,t=\Sigma_i w_i * r_i,t$
    
    - => realises the actual portfolio investing part

- <font color="green">evaluate_portfolio()</font>

    - takes returns and evaluates them
    - computes mean, std, psharpe and a couple more metrics annually (252 trading days in a year)

- <font color="green">plot_portfolios()</font>

    - visualizes the whole thing

**Welche Teile sind Datenaufbereitung/Datenverarbeitung?**

filter_universe()
metric_sharpe()
select_stocks()
panel_to_ret_matrix()

**Welche Teile sind Finance / Portfolio-Optimierung?**

covariance_estimator()
compute_weights() (naive, mcap, min_variance)
evaluate_portfolio

**Welche Rolle spielt `LedoitWolf`?**

method to estimate the covariance matrix

**Welche Rolle spielt `cvxpy`?**

solves the minimization problem when determining the min_variance weights

**Offene Fragen zu dieser Zelle:**

TODO

In [ ]:
# Experimentierfeld zu Zelle 1
# Idee: Hier kannst du einzelne Imports, Funktionen oder kleine Beispiele isoliert testen.

# Beispiel:
# import numpy as np
# import pandas as pd

c:\Users\phili\miniconda3\envs\phpo\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 1000/1000 [00:00<00:00, 999834.09it/s]


### Zelle 2: Abschnittsüberschrift zu Gewichten und Portfolio Returns

**Originaltyp:** Markdown

**Originalinhalt:** `## Determine weights and portfolio returns over time`

**Was wird in diesem Abschnitt vermutlich gemacht?**

TODO

**Warum sind Gewichte und Portfolio Returns zeitabhängig?**

TODO

**Begriffe, die hier wichtig werden:**

- Portfolio weights
- Holding period
- Rebalancing
- Rolling estimation window
- Out-of-sample returns

### Zelle 3: Datenimport, Parameter und Rebalancing-Schleife

**Originaltyp:** Code

**Erste sichtbare Bestandteile:**

- Daten werden aus `./Data/US_datastream/US_data_panel_filtered_0.15.feather` geladen
- Sortierung nach `Date`
- `estimation_window = 512`
- `holding_period = 21`
- Definition von Datums-, Return- und ID-Spalten
- Bestimmung von Rebalancing-Zeitpunkten
- Initialisierung von Investoren / Strategien

**Was macht die Zelle technisch?**

TODO

**Welche Daten werden verwendet?**

TODO: Was ist eine Feather-Datei? Was könnte in diesem Panel enthalten sein?

**Was bedeutet `estimation_window = 512` fachlich?**

TODO

**Was bedeutet `holding_period = 21` fachlich?**

TODO

**Welche Annahmen stecken in Rolling Window und Rebalancing?**

TODO

**Welche Outputs entstehen vermutlich?**

TODO

**Offene Fragen zu dieser Zelle:**

TODO

In [ ]:
# Experimentierfeld zu Zelle 3
# Idee: Lade bei Bedarf nur kleine Ausschnitte oder prüfe Spalten, Dimensionen und Datumsbereiche.

# Beispiel:
# import pandas as pd
# df = pd.read_feather('./Data/US_datastream/US_data_panel_filtered_0.15.feather')
# df.head()
# df.shape
# df.columns

### Zelle 4: Abschnittsüberschrift zu Summary und Visualisierung

**Originaltyp:** Markdown

**Originalinhalt:** `## Summary and Visualization`

**Was sollte eine gute Portfolio-Zusammenfassung enthalten?**

TODO

**Welche Kennzahlen könnten hier wichtig sein?**

TODO: Zum Beispiel durchschnittlicher Return, Volatilität, Sharpe Ratio, Drawdown, Turnover, Transaktionskosten.

### Zelle 5: Portfolio-Evaluation zusammenbauen

**Originaltyp:** Code

**Erste sichtbare Bestandteile:**

- `portfolio_evaluations = []`
- `portfolio_timeseries = []`
- Schleife über `investors.items()`
- `evaluate_portfolio(return_history)`
- Zusammenbau einer `summary`

**Was macht die Zelle technisch?**

TODO

**Welche Performance-Kennzahlen werden berechnet?**

TODO

**Was ist der Unterschied zwischen `portfolio_evaluations` und `portfolio_timeseries`?**

TODO

**Welche Finance-Frage beantwortet diese Zelle?**

TODO

**Offene Fragen zu dieser Zelle:**

TODO

In [ ]:
# Experimentierfeld zu Zelle 5
# Idee: Teste hier, wie eine einfache Return-Zeitreihe ausgewertet werden könnte.

# Beispiel:
# import pandas as pd
# returns = pd.Series([0.01, -0.005, 0.002, 0.004])
# returns.mean(), returns.std()

### Zelle 6: Visualisierung der Portfolio-Zeitreihen

**Originaltyp:** Code

**Originalinhalt:** `visualization = plot_portfolios(pd.concat(portfolio_timeseries, axis = 1))`

**Was macht die Zelle technisch?**

TODO

**Was wird wahrscheinlich geplottet?**

TODO

**Warum ist eine Visualisierung zusätzlich zur Summary-Tabelle nützlich?**

TODO

**Welche Risiken hat eine rein visuelle Interpretation?**

TODO

In [ ]:
# Experimentierfeld zu Zelle 6
# Idee: Erzeuge eine kleine Beispiel-Zeitreihe und visualisiere sie.

# Beispiel:
# import pandas as pd
# import matplotlib.pyplot as plt
# example = pd.Series([1.0, 1.01, 0.99, 1.03], name='Example portfolio')
# example.plot()
# plt.show()

### Zelle 7: Next possible steps

**Originaltyp:** Markdown

**Originalinhalt:** `## Nexts possible steps`

**Welche nächsten Schritte nennt oder impliziert das Original-Notebook?**

TODO

**Welche nächsten Schritte wären für die Bachelorarbeit sinnvoll?**

TODO

**Mögliche Kategorien:**

- Robustere Kovarianzschätzung
- Transaktionskosten
- Turnover
- Alternative Strategien / Benchmarks
- Out-of-sample Evaluation
- Sensitivitätsanalysen
- Datenqualität und Survivorship Bias

## 3. Begriffsliste

| Begriff | Eigene Erklärung | Wo taucht es im Notebook auf? | Noch unklar? |
|---|---|---|---|
| Minimum Variance Portfolio | TODO | TODO | TODO |
| Kovarianzmatrix | TODO | TODO | TODO |
| Kovarianzschätzung | TODO | TODO | TODO |
| Shrinkage | TODO | TODO | TODO |
| Ledoit-Wolf | TODO | TODO | TODO |
| Rolling Window | TODO | TODO | TODO |
| Estimation Window | TODO | TODO | TODO |
| Holding Period | TODO | TODO | TODO |
| Rebalancing | TODO | TODO | TODO |
| Turnover | TODO | TODO | TODO |
| Transaktionskosten | TODO | TODO | TODO |
| Out-of-sample Evaluation | TODO | TODO | TODO |
| Benchmark | TODO | TODO | TODO |
| Sharpe Ratio | TODO | TODO | TODO |
| Volatilität | TODO | TODO | TODO |

## 4. Annahmen im Notebook

Sammle hier alle Annahmen, die du beim Lesen findest.

| Annahme | Warum wird sie vermutlich gemacht? | Mögliche Auswirkung auf Ergebnisse |
|---|---|---|
| TODO | TODO | TODO |
| TODO | TODO | TODO |
| TODO | TODO | TODO |

## 5. Datenaufbereitung vs. Finance vs. Evaluation

Ordne Codebestandteile aus `min_variance.ipynb` einer Kategorie zu.

| Codebestandteil / Funktion | Kategorie | Warum? |
|---|---|---|
| TODO | Datenaufbereitung | TODO |
| TODO | Finance / Optimierung | TODO |
| TODO | Evaluation | TODO |
| TODO | Visualisierung | TODO |

## 6. Eigene Zusammenfassung

Schreibe am Ende eine eigene Zusammenfassung, ohne direkt in den Code zu schauen.

**In einem Satz:**

TODO

**In einem Absatz:**

TODO

**Was kann ich nach dieser Analyse besser erklären als vorher?**

TODO

**Welche 3 Fragen möchte ich als nächstes klären?**

1. TODO
2. TODO
3. TODO